# XAI-Compress Final Research Analysis
Evidence-driven notebook. Every benchmark figure is loaded from `results/final_analysis`; missing evidence remains **N/A**.

## Problem → theory → architecture
XAI-Compress models byte probabilities while XAIC v3 entropy coding preserves exact bytes. Shannon entropy is $H(X)=-\sum_x p(x)\log_2 p(x)$. The autoregressive factorization is $P(x_1,\ldots,x_n)=\prod_t P(x_t\mid x_{<t})$, giving ideal symbol length $L(x_t)\approx-\log_2P(x_t\mid x_{<t})$. Measured $BPB=8\,C/N$ (lower is better), while compression ratio $CR=N/C$ (higher is better).

## Pipeline and experiment evolution
Dataset bytes → bounded contexts → causal GRU/PyTorch CUDA → deterministic probabilities → arithmetic/rANS → XAIC v3 chunks → SHA-256 gate.

Observed project stages (no invented dates): initial GRU → improved training → XAIC v3 streaming → deterministic long-stream fix → Rust probability quantization → RESEARCH model → final benchmark.

In [ ]:
from pathlib import Path
import json, pandas as pd, matplotlib.pyplot as plt
ROOT=Path('..').resolve() if Path.cwd().name=='notebooks' else Path.cwd()
OUT=ROOT/'results'/'final_analysis'
REPORT=json.loads((OUT/'final_report.json').read_text())
print('Generated:',REPORT['generated_utc'])
print('Research status:',REPORT['research_training']['status'])
pd.read_csv(OUT/'model_inventory.csv')


## Dataset engineering
The benchmark manifest records category, exact byte size, SHA-256, source, and training-overlap status. Fallback project files are explicitly marked `UNKNOWN`, never assumed held out.

In [ ]:
manifest=pd.read_csv(OUT/'manifests'/'benchmark_manifest.csv'); display(manifest)
manifest.groupby('category').size().plot.bar(title='Dataset composition'); plt.show()
manifest['size'].plot.hist(bins=20,title='File-size distribution'); plt.show()


## Training and hardware evidence
Figures are generated only when synchronized metrics exist. The red dashed line identifies the measured best epoch.

In [ ]:
from IPython.display import display, Image, Markdown
for name in ['training_loss','validation_loss','bpb_evolution','learning_rate','gpu_utilization','vram_usage','training_throughput']:
 p=OUT/'figures'/f'{name}.png'
 if p.exists(): display(Markdown(f'### {name}')); display(Image(filename=str(p)))


## Rust optimization and Amdahl’s law
Rust accelerates the measured deterministic probability-quantization boundary, not GRU training. With optimized fraction $p$ and local speedup $s$, end-to-end speedup is $S=1/((1-p)+p/s)$. Therefore any local quantization speedup does **not** imply the same end-to-end compression speedup.

In [ ]:
display(pd.read_csv(OUT/'rust_vs_python.csv'))
for name in ['rust_speedup','rust_latency','rust_memory']:
 p=OUT/'figures'/f'{name}.png'
 if p.exists(): display(Image(filename=str(p)))


## Losslessness, model comparison and Pareto frontier
Only configurations passing every SHA-256 test are eligible. BPB, speed, and memory remain separate objectives; no arbitrary universal score is used.

In [ ]:
for file in ['roundtrip_results.csv','final_comparison.csv','results_by_data_type.csv']:
 p=OUT/file
 if p.exists(): display(Markdown(f'### {file}')); display(pd.read_csv(p))
for name in ['compression_ratio_comparison','bits_per_byte_comparison','compression_speed_comparison','decompression_speed_comparison','peak_memory_comparison','pareto_compression','pareto_decompression','input_size_vs_peak_rss','input_size_vs_compression_time']:
 p=OUT/'figures'/f'{name}.png'
 if p.exists(): display(Image(filename=str(p)))


## Limitations and next research
Already-compressed media is not transformed to favor the neural method. Checkpoint distribution overhead must be reported separately from payload size. Streaming memory conclusions require isolated-process RSS measurements across real increasing file sizes. The next experiment should target whichever measured component dominates end-to-end latency after RESEARCH completes.

In [ ]:
display(Markdown((OUT/'conclusion.md').read_text()))
